# Task 2.11 — Submission DL từ checkpoint đã chọn

**Inputs bắt buộc:** competition data + outputs **2.6, 2.9 và 2.10**, đúng các version đã dùng với nhau.
Notebook **không train lại**: mỗi config dùng trung bình dự đoán của 5 checkpoint fold; sau đó kết hợp các config bằng trọng số đã chốt ở 2.10.
Mỗi checkpoint nạp đúng preprocessor/config đã fit trên training fold, không fit gì trên test.
Có thể chạy riêng notebook này sau khi đủ các output; không tự submit leaderboard.


In [ ]:
from pathlib import Path
import importlib.util
import sys
import hashlib
import json

DATA_DIR = None  # Auto-detect inputs; set a directory here only to disambiguate.

def resolve_data_dir(data_dir=None, expected_sha256=None,
                     required_files=("train.csv",), input_root="/kaggle/input"):
    """Find attached data without assuming Kaggle's mount layout."""
    from pathlib import Path
    import hashlib

    root = Path(input_root)
    if data_dir is not None:
        candidates = [Path(data_dir)]
        missing = [name for name in required_files if not (candidates[0] / name).is_file()]
        if missing:
            raise FileNotFoundError(
                f"DATA_DIR={data_dir} is missing {missing}. "
                "Attach competition data or set DATA_DIR=None for automatic discovery."
            )
    else:
        candidates = sorted({
            path.parent for path in root.rglob("train.csv")
            if path.is_file() and all((path.parent / name).is_file() for name in required_files)
        })
        if not candidates:
            mounted = sorted(path.name for path in root.iterdir()) if root.is_dir() else []
            raise FileNotFoundError(
                f"No attached directory contains {list(required_files)} under {root}. "
                "Use Add Input to attach competition playground-series-s6e11. "
                f"Mounted entries: {mounted}"
            )

    if expected_sha256 is not None:
        matching = []
        for directory in candidates:
            digest = hashlib.sha256()
            with (directory / "train.csv").open("rb") as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b""):
                    digest.update(block)
            if digest.hexdigest() == expected_sha256:
                matching.append(directory)
        if not matching:
            raise ValueError(
                "Attached train.csv does not match the task 2.6 checksum. "
                f"Check the competition data/version. Candidates: {candidates}"
            )
        candidates = matching
    elif len(candidates) > 1:
        bundles = [directory for directory in candidates
                   if all((directory / name).is_file()
                          for name in ("train.csv", "test.csv", "sample_submission.csv"))]
        if bundles:
            candidates = bundles

    if len(candidates) != 1:
        raise ValueError(
            f"Multiple valid data directories found: {candidates}. "
            "Set DATA_DIR to the exact directory you want to use."
        )
    return candidates[0]

# Nếu gắn nhiều version 2.6, điền chính xác thư mục version cần dùng.
FOUNDATION_DIR = None
if FOUNDATION_DIR is None:
    found = sorted(Path("/kaggle/input").rglob("foundation_manifest.json"))
    if len(found) != 1:
        raise FileNotFoundError(f"Attach exactly one task 2.6 output; found: {found}")
    FOUNDATION_DIR = found[0].parent
else:
    FOUNDATION_DIR = Path(FOUNDATION_DIR)
manifest = json.loads((FOUNDATION_DIR / "foundation_manifest.json").read_text(encoding="utf-8"))
engine_file = FOUNDATION_DIR / "credit_fraud_dl_engine.py"
if hashlib.sha256(engine_file.read_bytes()).hexdigest() != manifest["engine_sha256"]:
    raise ValueError("Foundation engine checksum mismatch.")
spec = importlib.util.spec_from_file_location("credit_fraud_dl_engine", engine_file)
dl = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = dl
spec.loader.exec_module(dl)

import numpy as np
import pandas as pd
DATA_DIR = resolve_data_dir(
    DATA_DIR, expected_sha256=manifest["train_sha256"],
    required_files=("train.csv", "test.csv", "sample_submission.csv"),
)
print("Resolved DATA_DIR:", DATA_DIR)
train_path = DATA_DIR / "train.csv"
dl.verify_source(train_path, manifest)
train = pd.read_csv(train_path)
dl.validate_train(train)
for name, digest in manifest["membership_sha256"].items():
    if dl.file_hash(FOUNDATION_DIR / name) != digest:
        raise ValueError(f"Foundation membership file changed: {name}")
split = pd.read_csv(FOUNDATION_DIR / "holdout_split_seed9.csv")
train_idx, valid_idx = dl.holdout_indices(train, split)
print("Foundation:", manifest["foundation_id"])
print("Runtime:", dl.environment())


## 2.11.1 — Kiểm tra nguồn quyết định và artifact

Chặn dùng nhầm version CV/selection hoặc thay engine sau training.
Tên mô hình không đủ để dựng lại mạng: inference đọc kích thước lớp, preprocessing và state_dict từ checkpoint/metadata thực tế.
Giữ nguyên các artifact 2.1–2.5 làm mốc lịch sử.


In [ ]:
OUTPUT_DIR = Path("/kaggle/working/task_2_11_optimized_submission")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cv_path = dl.find_input("cv_manifest.json")
cv = dl.json_read(cv_path)
selection_path = dl.find_input("ensemble_selection.json")
selection = dl.json_read(selection_path)
if (selection["foundation_id"] != manifest["foundation_id"]
        or cv["foundation_id"] != manifest["foundation_id"]
        or selection["source_sha256"] != manifest["train_sha256"]):
    raise ValueError("Different foundation or training dataset.")
if selection["cv_manifest_sha256"] != dl.file_hash(cv_path):
    raise ValueError("Selection references a different CV version.")
if selection["engine_sha256"] != manifest["engine_sha256"]:
    raise ValueError("Inference engine differs from the selected models.")
weights = selection["weights"]
if (not weights or not np.isclose(sum(weights.values()), 1.0)
        or any(not np.isfinite(w) or w < 0 for w in weights.values())):
    raise ValueError("Invalid frozen blend weights.")
for key in weights:
    if key not in cv["models"] or selection["models"][key] != cv["models"][key]:
        raise ValueError("Selection model metadata differs from CV output.")

test = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
if dl.TARGET in test or dl.ID not in test:
    raise ValueError("Test must contain ID but no target labels.")
raw_test = test.drop(columns=[dl.ID])
if raw_test.columns.tolist() != train.drop(columns=[dl.ID, dl.TARGET]).columns.tolist():
    raise ValueError("Train/test feature schema differs.")


## 2.11.2 — Trung bình 5-fold, rồi blend các config

Không chọn fold có điểm cao nhất. Mỗi fold có một vote bằng nhau trong config; trọng số giữa config được khóa từ OOF.
Dự đoán theo batch và giải phóng model sau từng fold. Ghi xác suất từng config để kiểm tra nguồn gốc.
Nếu checkpoint là TabNet, cần đúng package đã dùng lúc train.


In [ ]:
test_probability = np.zeros(len(test), dtype=np.float64)
members = []
for key, weight in weights.items():
    model = cv["models"][key]
    fold_paths = model["fold_paths"]
    if len(fold_paths) != 5 or len(set(fold_paths)) != 5:
        raise ValueError("Expected five distinct checkpoints per config.")
    config_probability = np.zeros(len(test), dtype=np.float64)
    if [item["run_directory"] for item in model["fold_artifacts"]] != fold_paths:
        raise ValueError("Frozen fold artifact list differs from checkpoint paths.")
    for artifact in model["fold_artifacts"]:
        relative_dir = artifact["run_directory"]
        run_dir = cv_path.parent / relative_dir
        if dl.file_hash(run_dir / "metrics.json") != artifact["metrics_sha256"]:
            raise ValueError("Fold metadata changed after CV model selection.")
        run_metadata = dl.json_read(run_dir / "metrics.json")
        if (not run_metadata.get("reload_verified", False)
                or run_metadata["run_signature"] != artifact["run_signature"]):
            raise ValueError("Unverified or different fold checkpoint.")
        if run_metadata["config"] != model["config"]:
            raise ValueError("Fold config differs from selected config.")
        if run_metadata["source_sha256"] != manifest["train_sha256"]:
            raise ValueError("Fold was trained with another dataset.")
        config_probability += dl.predict_run(run_dir, raw_test) / len(fold_paths)
        members.append({
            "config_id": key, "fold_directory": relative_dir,
            "blend_weight": weight / len(fold_paths),
            "run_signature": run_metadata["run_signature"],
            "checkpoint_sha256": run_metadata["artifact_sha256"][run_metadata["model_file"]],
            "best_epoch": run_metadata["best_epoch"],
        })
    config_probability = dl.check_probability(config_probability, len(test))
    pd.DataFrame({dl.ID: test[dl.ID], "probability": config_probability}).to_csv(
        OUTPUT_DIR / f"{key}_test_predictions.csv", index=False)
    test_probability += weight * config_probability

submission = dl.export_submission(
    test, sample, test_probability, OUTPUT_DIR / "submission.csv")
display(submission.head())
print("Rows:", len(submission), "Probability range:",
      submission[dl.TARGET].min(), submission[dl.TARGET].max())


## 2.11.3 — Metadata và báo cáo

Điểm 0.94263 là điểm baseline bạn báo trước đây. Notebook mới chưa có điểm Kaggle cho đến khi người dùng nộp CSV.
Lưu mean fold AUC và pooled OOF riêng; không gọi CV là kết quả test hoặc đảm bảo điểm leaderboard tăng.


In [ ]:
metadata = {
    "foundation_id": manifest["foundation_id"],
    "train_sha256": manifest["train_sha256"],
    "test_sha256": dl.file_hash(DATA_DIR / "test.csv"),
    "selection_sha256": dl.file_hash(selection_path),
    "cv_manifest_sha256": dl.file_hash(cv_path),
    "engine_sha256": manifest["engine_sha256"],
    "weights": weights, "members": members,
    "configurations": {key: cv["models"][key]["config"] for key in weights},
    "validation": selection["selected_metrics"],
    "baseline_validation": selection["baseline_metrics"],
    "inference_environment": dl.environment(),
    "submission_rows": len(submission), "columns": submission.columns.tolist(),
    "submission_sha256": dl.file_hash(OUTPUT_DIR / "submission.csv"),
    "historical_baseline_public_score_user_reported": 0.94263,
    "new_leaderboard_score": None, "competition_submission_uploaded": False,
    "test_labels_used": False,
    "limitations": selection["limitations"],
}
dl.json_write(OUTPUT_DIR / "submission_metadata.json", metadata)
report = (
    "# Kết quả pipeline tối ưu DL\n\n"
    f"- Trọng số config: {weights}\n"
    f"- Mean fold ROC-AUC: {selection['selected_metrics']['mean_fold_auc']:.6f}\n"
    f"- Pooled OOF ROC-AUC: {selection['selected_metrics']['pooled_oof']['roc_auc']:.6f}\n"
    f"- Mean fold baseline: {selection['baseline_metrics']['mean_fold_auc']:.6f}\n"
    f"- Submission: {len(submission):,} dòng; xác suất is_fraud, ghép theo ID.\n"
    "- Sử dụng checkpoint 5-fold, không refit toàn bộ train và không train thêm ở notebook này.\n"
    "- Baseline public score 0.94263 do người dùng báo; chưa có leaderboard score mới.\n"
    "- CV/OOF đã dùng để chọn mô hình và trọng số, không phải test độc lập.\n"
    "- Chưa tự submit Kaggle.\n"
)
(OUTPUT_DIR / "optimization_report.md").write_text(report, encoding="utf-8")
print("Download submission.csv after Save Version completes successfully.")
